In [39]:
import pandas as pd
import numpy as np
import json

def log_interactive_js_table_to_wandb(
    df: pd.DataFrame,
    table_name: str = "validation_results_js_interactive",
):
    """
    Generuje a loguje plně interaktivní, bezserverový HTML report.
    FINÁLNÍ VERZE: Opravuje problém s časováním předáním API jako parametru.
    """

    # Automatické přejmenování 'target' na 'actual_fitness' pro kompatibilitu
    if 'target' in df.columns and 'actual_fitness' not in df.columns:
        print("INFO: Přejmenovávám sloupec 'target' na 'actual_fitness'.")
        df = df.rename(columns={'target': 'actual_fitness'})

    # Kontrola, zda máme vše potřebné
    required_cols = ['actual_fitness', 'predicted_fitness']
    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"CHYBA: V DataFrame chybí klíčové sloupce: {required_cols}. Metriky nelze spočítat.")

    if 'error' not in df.columns:
        df['error'] = df["predicted_fitness"] - df["actual_fitness"]

    # Čištění dat pro JSON
    df_clean = df.where(pd.notnull(df), None)
    df_json = df_clean.to_json(orient='records')

    html_template = f"""
<!DOCTYPE html>
<html lang="cs">
<head>
    <meta charset="UTF-8">
    <title>Interactive Validation Results</title>
    <script src="https://cdn.jsdelivr.net/npm/ag-grid-community/dist/ag-grid-community.min.js"></script>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/ag-grid-community/styles/ag-grid.css" />
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/ag-grid-community/styles/ag-theme-alpine.css" />
    <style>
        body {{ font-family: sans-serif; padding: 20px; }}
        .metric-container {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 15px; margin-bottom: 20px; }}
        .metric-card {{ padding: 15px; border: 1px solid #ddd; border-radius: 5px; background-color: #f9f9f9; }}
        .metric-title {{ font-weight: bold; color: #333; }}
        .metric-value {{ font-size: 1.2em; color: #0056b3; font-family: monospace; }}
        #myGrid {{ height: 600px; width: 100%; }}
    </style>
</head>
<body>
    <h1>Interaktivní analýza výsledků</h1>
    <div class="metric-container">
        <div class="metric-card"><span class="metric-title">Zobrazeno vzorků:</span> <span id="samples-value" class="metric-value">-</span></div>
        <div class="metric-card"><span class="metric-title">Platných pro výpočet:</span> <span id="valid-samples-value" class="metric-value">-</span></div>
        <div class="metric-card"><span class="metric-title">MSE:</span> <span id="mse-value" class="metric-value">-</span></div>
        <div class="metric-card"><span class="metric-title">MAE:</span> <span id="mae-value" class="metric-value">-</span></div>
        <div class="metric-card"><span class="metric-title">RMSE:</span> <span id="rmse-value" class="metric-value">-</span></div>
        <div class="metric-card"><span class="metric-title">R²:</span> <span id="r2-value" class="metric-value">-</span></div>
        <div class="metric-card"><span class="metric-title">Pearson Corr:</span> <span id="pearson-value" class="metric-value">-</span></div>
    </div>

    <div id="myGrid" class="ag-theme-alpine"></div>

    <script>
        const rowData = {df_json};

        const columnDefs = Object.keys(rowData[0] || {{}}).map(key => ({{
            field: key,
            filter: typeof rowData[0][key] === 'number' ? 'agNumberColumnFilter' : 'agTextColumnFilter',
            sortable: true, resizable: true
        }}));

        const gridOptions = {{
            columnDefs: columnDefs,
            rowData: rowData,
            defaultColDef: {{ flex: 1, minWidth: 120, filter: true, sortable: true, resizable: true }},

            // ========================= ZMĚNA ZDE =========================
            onFilterChanged: (params) => updateAllMetrics(params.api),
            onFirstDataRendered: (params) => updateAllMetrics(params.api),
            // =============================================================
        }};

        function calculateMetrics(data) {{
            if (!data || data.length < 2 || !data[0].hasOwnProperty('actual_fitness')) {{
                return {{ samples: data ? data.length : 0, valid: 0 }};
            }}

            const validPairs = data
                .map(row => ({{
                    t: parseFloat(row.actual_fitness),
                    p: parseFloat(row.predicted_fitness)
                }}))
                .filter(pair => !isNaN(pair.t) && !isNaN(pair.p));

            const n = validPairs.length;
            if (n < 2) return {{ samples: data.length, valid: n }};

            let sum_sq_err = 0, sum_abs_err = 0, sum_true = 0, total_sum_sq = 0;
            let sum_xy = 0, sum_x = 0, sum_y = 0, sum_x2 = 0, sum_y2 = 0;

            validPairs.forEach(pair => {{
                sum_true += pair.t;
                sum_x += pair.t; sum_y += pair.p; sum_xy += pair.t * pair.p;
                sum_x2 += pair.t * pair.t; sum_y2 += pair.p * pair.p;
            }});
            const mean_true = sum_true / n;

            validPairs.forEach(pair => {{
                sum_sq_err += (pair.p - pair.t) ** 2;
                sum_abs_err += Math.abs(pair.p - pair.t);
                total_sum_sq += (pair.t - mean_true) ** 2;
            }});

            const mse = sum_sq_err / n;
            const r2 = total_sum_sq < 1e-9 ? 1 : 1 - (sum_sq_err / total_sum_sq);
            const num = n * sum_xy - sum_x * sum_y;
            const den = Math.sqrt((n * sum_x2 - sum_x**2) * (n * sum_y2 - sum_y**2));
            const pearson = den < 1e-9 ? 0 : num / den;

            return {{
                samples: data.length, valid: n, mse: mse,
                mae: sum_abs_err / n, rmse: Math.sqrt(mse), r2: r2, pearson: pearson
            }};
        }}

        // ========================= ZMĚNA ZDE =========================
        function updateAllMetrics(api) {{
        // =============================================================
            const rows = [];
            if (api) {{
                api.forEachNodeAfterFilter(node => rows.push(node.data));
            }}

            const metrics = calculateMetrics(rows);

            document.getElementById('samples-value').innerText = metrics.samples;
            document.getElementById('valid-samples-value').innerText = metrics.valid;

            ['mse', 'mae', 'rmse', 'r2', 'pearson'].forEach(key => {{
                 const el = document.getElementById(key + '-value');
                 if (el) {{
                     const value = metrics[key];
                     el.innerText = (value === undefined || isNaN(value)) ? '-' : value.toFixed(6);
                 }}
            }});
        }}

        document.addEventListener('DOMContentLoaded', () => {{
            const gridDiv = document.querySelector('#myGrid');
            agGrid.createGrid(gridDiv, gridOptions);
        }});
    </script>
</body>
</html>
    """

    local_filename = f"{table_name}_final_working.html"
    with open(local_filename, "w", encoding="utf-8") as f:
        f.write(html_template)
    print(f"Finální funkční soubor uložen jako: {local_filename}")

    return html_template


In [2]:
import pandas as pd
from bs4 import BeautifulSoup

def html_table_to_dataframe(html_code: str) -> pd.DataFrame:
    """
    Parsování HTML kódu z knihovny Tabulator.js a jeho převod na pandas DataFrame.

    Args:
        html_code: Řetězec obsahující HTML kód tabulky.

    Returns:
        pandas DataFrame s extrahovanými daty.
    """
    soup = BeautifulSoup(html_code, 'html.parser')

    # 1. Extrakce názvů sloupců (hlavičky)
    header_divs = soup.select('.tabulator-header .tabulator-col')
    headers = [
        header.get('tabulator-field')
        for header in header_divs
        if header.get('tabulator-field') and not header.get('tabulator-field').startswith('_') and not header.has_attr('empty')
    ]

    # 2. Extrakce dat z jednotlivých řádků
    all_rows_data = []
    rows = soup.select('.tabulator-table .tabulator-row')

    for row in rows:
        row_data = {}
        cells = row.select('.tabulator-cell')
        for cell in cells:
            field_name = cell.get('tabulator-field')
            if field_name in headers:
                # Získání textu z vnořeného divu, odstranění bílých znaků
                cell_value = cell.get_text(strip=True)
                row_data[field_name] = cell_value

        if row_data:
            all_rows_data.append(row_data)

    # 3. Vytvoření DataFrame
    if not all_rows_data:
        return pd.DataFrame(columns=headers) # Vrátí prázdný DF, pokud se nic nenašlo

    df = pd.DataFrame(all_rows_data)

    # Zajistí správné pořadí sloupců, pokud by se nějaký řádek lišil
    df = df[headers]

    # 4. Převod datových typů a čištění
    for col in df.columns:
        # Pokus o převod na numerické hodnoty tam, kde je to možné
        df[col] = pd.to_numeric(df[col], errors='ignore')

    # Speciální ošetření pro booleovský sloupec 'reverse'
    if 'reverse' in df.columns:
        # Převede 'true' na True a 'false' na False, ostatní hodnoty nechá beze změny
        df['reverse'] = df['reverse'].apply(lambda x: True if str(x).lower() == 'true' else (False if str(x).lower() == 'false' else x))
        # Převede sloupec na nullable boolean typ pro konzistenci
        df['reverse'] = df['reverse'].astype('boolean')

    return df

# --- POUŽITÍ ---

# Vložte sem svůj kompletní HTML kód
html_code = """
<div style="width: 100%; height: 100%; visibility: visible;" class="pnx-tabulator tabulator" role="grid" tabulator-layout="fitColumns"><div class="tabulator-header" role="rowgroup"><div class="tabulator-header-contents" role="rowgroup"><div class="tabulator-headers" role="row" style="height: 58px;"><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element tabulator-frozen tabulator-frozen-left" role="columnheader" aria-sort="none" style="display: none; min-width: 40px; position: sticky; left: 0px; height: 58px;" tabulator-field="_index"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">&nbsp;</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div></div></div><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="index" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">index</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="number" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="original_seq_full" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">original_seq_full</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="search" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="mutated_seq_full" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">mutated_seq_full</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="search" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="mut_type" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">mut_type</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="search" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="target" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">target</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="number" step="0.01" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="reverse" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">reverse</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="checkbox" style="margin-top: 5px; box-sizing: border-box;" value="" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="data_source" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">data_source</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="search" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="fragment_255_mut" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">fragment_255_mut</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="search" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="fragment_255_org" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">fragment_255_org</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="search" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="predicted_fitness" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">predicted_fitness</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="number" step="0.01" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="actual_fitness" style="min-width: 40px; width: 165px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">actual_fitness</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="number" step="0.01" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element" role="columnheader" aria-sort="none" tabulator-field="error" style="min-width: 40px; width: 166px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">error</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div><div class="tabulator-header-filter"><input type="number" step="0.01" style="padding: 4px; width: 100%; box-sizing: border-box;" placeholder=""></div></div></div><span class="tabulator-col-resize-handle" style="height: 58px;"></span><div class="tabulator-col tabulator-sortable tabulator-col-sorter-element empty" role="columnheader" aria-sort="none" style="min-width: 1px; max-width: 1px; width: 1px; height: 58px;"><div class="tabulator-col-content"><div class="tabulator-col-title-holder"><div class="tabulator-col-title">&nbsp;</div><div class="tabulator-col-sorter"><div class="tabulator-arrow"></div></div></div></div></div></div><br><div class="tabulator-frozen-rows-holder" style="min-width: 1982px;"></div></div></div><div class="tabulator-tableholder" tabindex="0" style="height: 580px;"><div class="tabulator-table" role="rowgroup" style="padding-top: 0px; padding-bottom: 0px;"><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">0</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">0</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGGSAGGSAGGSDLRKKIVDLHKSGSSLGAISKRLKVPRSSVDTIVRKYKSAGGSAGGSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGGSAGGSAGGSDLRKKIVDLHKSGSSLGAISKRLKVPRSSVQTIVRKYKSAGGSAGGSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">Q31D</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">0.117479</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">SAGGSAGGSAGGSAGGSDLRKKIVDLHKSGSSLGAISKRLKVPRSSVQTIVRKYKSAGGSAGGSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">SAGGSAGGSAGGSAGGSDLRKKIVDLHKSGSSLGAISKRLKVPRSSVDTIVRKYKSAGGSAGGSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.032819</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">0.117479</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.150298</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">1</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">1</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MPSGSSAALALAAAPAPLPQPPPPPPPPPPPLPPPSGGPELEGDGLLLRERLAALGLDDPSPAEPGAPALRAPAAAAQGQARRAAELSPEERAPPGRPGAPEAAELELEEDEEEGEEAELDGDLLEEEELEEAEEEDRSSLLLLSPPAATASQTQQIPGGSLGSVLLPAARFDAREAAAAAAAAGVLYGGDDAQGMMAAMLSHAYGPGGCGAAAAALNGEQAALLRRKSVNTTECVPVPSSEHVAEIVGRQGCKIKALRAKTNTYIKTPVRGEEPIFVVTGRKEDVAMAKREILSAAEHFSMIRASRNKNGPALGGLSCSPNLPGQTTVQVRVPYRVVGLVVGPKGATIKRIQQQTHTYIVTPSRPKEPVFEVTGMPENVDRAREEIEMHIAMRTGNYIELNEENDFHYNGTDVSFEGGTLGSAWLSSNPVPPSRARMISNYRNDSSSSLGSGSTDSYFGSNRLADFSPTSPFSTGNFWFGDTLPSVGSEDLAVDSPAFDSLPTSAQTIWTPFEPVNPLSGFGSDPSGNMKTQRRGSQPSTPRLSPTFPESIEHPLARRVRSDPPSTGNHVGLPIYIPAFSNGTNSYSSSNGGSTSSSPPESRRKHDCVICFENEVIAALVPCGHNLFCMECANKICEKRTPSCPVCQTAVTQAIQIHS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MPSGSSAALALAAAPAPLPQPPPPPPPPPPPLPPPSGGPELEGDGLLLRERLAALGLDDPSPAEPGAPALRAPAAAAQGQARRAAELSPEERAPPGRPGAPEAAELELEEDEEEGEEAELDGDLLEEEELEEAEEEDRSSLLLLSPPAATASQTQQIPGGSLGSVLLPAARFDAREAAAAAAAAGVLYGGDDAQGMMAAMLSHAYGPGGCGAAAAALNGEQAALLRRKSVNTTECVPVPSSEHVAEIVGRQGCKIKALRAKTNTYIKTPVRGEEPIFVVTGRKEDVAMAKREILSAAEHFSMIRASRNKNGPALGGLSCSPNLPGQTTVQVRVPYRVVGLVVGPKGATIKRIQQQTHTYIVTPSRDKEPVFEVTGMPENVDRAREEIEMHIAMRTGNYIELNEENDFHYNGTDVSFEGGTLGSAWLSSNPVPPSRARMISNYRNDSSSSLGSGSTDSYFGSNRLADFSPTSPFSTGNFWFGDTLPSVGSEDLAVDSPAFDSLPTSAQTIWTPFEPVNPLSGFGSDPSGNMKTQRRGSQPSTPRLSPTFPESIEHPLARRVRSDPPSTGNHVGLPIYIPAFSNGTNSYSSSNGGSTSSSPPESRRKHDCVICFENEVIAALVPCGHNLFCMECANKICEKRTPSCPVCQTAVTQAIQIHS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">D366P</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">0.414292</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">PSSEHVAEIVGRQGCKIKALRAKTNTYIKTPVRGEEPIFVVTGRKEDVAMAKREILSAAEHFSMIRASRNKNGPALGGLSCSPNLPGQTTVQVRVPYRVVGLVVGPKGATIKRIQQQTHTYIVTPSRDKEPVFEVTGMPENVDRAREEIEMHIAMRTGNYIELNEENDFHYNGTDVSFEGGTLGSAWLSSNPVPPSRARMISNYRNDSSSSLGSGSTDSYFGSNRLADFSPTSPFSTGNFWFGDTLPSVGSEDLA</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">PSSEHVAEIVGRQGCKIKALRAKTNTYIKTPVRGEEPIFVVTGRKEDVAMAKREILSAAEHFSMIRASRNKNGPALGGLSCSPNLPGQTTVQVRVPYRVVGLVVGPKGATIKRIQQQTHTYIVTPSRPKEPVFEVTGMPENVDRAREEIEMHIAMRTGNYIELNEENDFHYNGTDVSFEGGTLGSAWLSSNPVPPSRARMISNYRNDSSSSLGSGSTDSYFGSNRLADFSPTSPFSTGNFWFGDTLPSVGSEDLA</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.022316</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">0.414292</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.391976</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">2</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">2</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGGKFNKELSVAGREIVTLPNLNDPQKKAFIFSLWDDPSQSANLLAEAKKLNDAQAPKSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGGKFNKELSVAGREIVTLPNLNDPQKKAFIFSLWDWPFQSANLLAEAKKLNDAQAPKSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">D34W:S36F</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.171998</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">SAGGSAGGKFNKELSVAGREIVTLPNLNDPQKKAFIFSLWDWPFQSANLLAEAKKLNDAQAPKSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">SAGGSAGGKFNKELSVAGREIVTLPNLNDPQKKAFIFSLWDDPSQSANLLAEAKKLNDAQAPKSAGGSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.191075</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.171998</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.019076</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">3</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">3</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MAALSGGGGGGAEPGQALFNGDMEPEAGAGAGAAASSAADPAIPEEVWNIKQMIKLTQEHIEALLDKFGGEHNPPSIYLEAYEEYTSKLDALQQREQQLLESLGNGTDFSVSSSASMDTVTSSSSSSLSVLPSSLSVFQNPTDVARSNPKSPQKPIVRVFLPNKQRTVVPARCGVTVRDSLKKALMMRGLIPECCAVYRIQDGEKKPIGWDTDISWLTGEELHVEVLENVPLTTHNFVRKTFFTLAFCDFCRKLLFQGFRCQTCGYKFHQRCSTEVPLMCVNYDQLDLLFVSKFFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPSPSKSIPIPQPFRPADEDHRNQFGQRDRSSSAPNVHINTIEPVNIDDLIRDQGFRGDGGSTTGLSATPPASLPGSLTNVKALQKSPGPQRERKSSSSSEDRNRMKTLGRRDSSDDWEIPDGQITVGQRIGSGSFGTVYKGKWHGDVAVKMLNVTAPTPQQLQAFKNEVGVLRKTRHVNILLFMGYSTKPQLAIVTQWCEGSSLYHHLHIIETKFEMIKLIDIARQTAQGMDYLHAKSIIHRDLKSNNIFLHEDLTVKIGDFGLATVKSRWSGSHQFEQLSGSILWMAPEVIRMQDKNPYSFQSDVYAFGIVLYELMTGQLPYSNINNRDQIIFMVGRGYLSPDLSKVRSNCPKAMKRLMAECLKKKRDERPLFPQILASIELLARSLPKIHRSASEPSLNRAGFQTEDFSLYACASPKTPIQAGGYGAFPVH</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MAALSGGGGGGAEPGQALFNGDMEPEAGAGAGAAASSAADPAIPEEVWNIKQMIKLTQEHIEALLDKFGGEHNPPSIYLEAYEEYTSKLDALQQREQQLLESLGNGTDFSVSSSASMDTVTSSSSSSLSVLPSSLSVFQNPTDVARSNPKSPQKPIVRVFLPNKQRTVVPARCGVTVRDSLKKALMMRGLIPECCAVYRIQDGEKKPIGEDTDISWLTGEELHVEVLENVPLTTHNFVRKTFFTLAFCDFCRKLLFQGFRCQTCGYKFHQRCSTEVPLMCVNYDQLDLLFVSKFFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPSPSKSIPIPQPFRPADEDHRNQFGQRDRSSSAPNVHINTIEPVNIDDLIRDQGFRGDGGSTTGLSATPPASLPGSLTNVKALQKSPGPQRERKSSSSSEDRNRMKTLGRRDSSDDWEIPDGQITVGQRIGSGSFGTVYKGKWHGDVAVKMLNVTAPTPQQLQAFKNEVGVLRKTRHVNILLFMGYSTKPQLAIVTQWCEGSSLYHHLHIIETKFEMIKLIDIARQTAQGMDYLHAKSIIHRDLKSNNIFLHEDLTVKIGDFGLATVKSRWSGSHQFEQLSGSILWMAPEVIRMQDKNPYSFQSDVYAFGIVLYELMTGQLPYSNINNRDQIIFMVGRGYLSPDLSKVRSNCPKAMKRLMAECLKKKRDERPLFPQILASIELLARSLPKIHRSASEPSLNRAGFQTEDFSLYACASPKTPIQAGGYGAFPVH</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">E210W</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.055847</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">EEYTSKLDALQQREQQLLESLGNGTDFSVSSSASMDTVTSSSSSSLSVLPSSLSVFQNPTDVARSNPKSPQKPIVRVFLPNKQRTVVPARCGVTVRDSLKKALMMRGLIPECCAVYRIQDGEKKPIGEDTDISWLTGEELHVEVLENVPLTTHNFVRKTFFTLAFCDFCRKLLFQGFRCQTCGYKFHQRCSTEVPLMCVNYDQLDLLFVSKFFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPSPS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">EEYTSKLDALQQREQQLLESLGNGTDFSVSSSASMDTVTSSSSSSLSVLPSSLSVFQNPTDVARSNPKSPQKPIVRVFLPNKQRTVVPARCGVTVRDSLKKALMMRGLIPECCAVYRIQDGEKKPIGWDTDISWLTGEELHVEVLENVPLTTHNFVRKTFFTLAFCDFCRKLLFQGFRCQTCGYKFHQRCSTEVPLMCVNYDQLDLLFVSKFFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPSPS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.072226</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.055847</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.128073</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">4</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">4</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MEAERRRQAEKPKKGRVGSNLLPERHPATGTPTTTVDSSAPPCRRLPGAGGGRSRFSPQGGQRGRPHSRRRHRTTFSPVQLEQLESAFGRNQYPDIWARESLARDTGLSEARIVVWFQNRRAKQRKQERSLLQPLAHLSPAAFSSFLPESTACPYSYAAPPPPVTCFPHPYSHALPSQPSTGGAFALSHQSEDWYPTLHPAPAGHLPCPPPPPMLPLSLEPSKSWN</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MEAERRRQAEKPKKGRVGSNLLPERHPATGTPTTTVDSSAPPCRRLPGAGGGRSRFSPQGGQRGRPHSRRRHRTTFSPVQLEQLESAFGRNQYPDIWARESLARDTGLSEARIQVWFQNRRAKQRKQERSLLQPLAHLSPAAFSSFLPESTACPYSYAAPPPPVTCFPHPYSHALPSQPSTGGAFALSHQSEDWYPTLHPAPAGHLPCPPPPPMLPLSLEPSKSWN</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">Q114V</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">0.046613</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">MEAERRRQAEKPKKGRVGSNLLPERHPATGTPTTTVDSSAPPCRRLPGAGGGRSRFSPQGGQRGRPHSRRRHRTTFSPVQLEQLESAFGRNQYPDIWARESLARDTGLSEARIQVWFQNRRAKQRKQERSLLQPLAHLSPAAFSSFLPESTACPYSYAAPPPPVTCFPHPYSHALPSQPSTGGAFALSHQSEDWYPTLHPAPAGHLPCPPPPPMLPLSLEPSKSWN</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">MEAERRRQAEKPKKGRVGSNLLPERHPATGTPTTTVDSSAPPCRRLPGAGGGRSRFSPQGGQRGRPHSRRRHRTTFSPVQLEQLESAFGRNQYPDIWARESLARDTGLSEARIVVWFQNRRAKQRKQERSLLQPLAHLSPAAFSSFLPESTACPYSYAAPPPPVTCFPHPYSHALPSQPSTGGAFALSHQSEDWYPTLHPAPAGHLPCPPPPPMLPLSLEPSKSWN</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.000619</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">0.046613</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.047232</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">5</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">5</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MSKRPSYAPPPTPAPATQMPSTPGFVGYNPYSHLAYNNYRLGGNPGTNSRVTASSGITIPKPPKPPDKPLMPYMRYSRKVWDQVKASNPDHKLWEIGKIIGGMWRDLTDEEKQEYLNEYEAEKIEYNESMKAYHNSPAYLAYINAKSRAEAALEEESRQRQSRMEKGEPYMSIQPAEDPDDYDDGFSMKHTATARFQRNHRLISEILSESVVPDVRSVVTTARMQVLKRQVQSLMVHQRKLEAELLQIEERHQEKKRKFLESTDSFNNELKRLCGLKVEVDMEKIAAEIAQAEEQARKRQEEREKEAAEQAERSQSSIVPEEEQAANKGEEKKDDENIPMETEETHLEETTESQQNGEEGTSTPEDKESGQEGVDSMAEEGTSDSNTGSESNSATVEEPPTDPIPEDEKKE</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MSKRPSYAPPPTPAPATQMPSTPGFVGYNPYSHLAYNNYRLGGNPGTNSRVTASSGITIPKPPKPPDKPLMPYMRYSRKVWDQVKASNPDLKLWEIGKIIGGMWRDLTDEEKQEYLNEYEAEKIEYNESMKAYHNSPAYLAYINAKSRAEAALEEESRQRQSRMEKGEPYMSIQPAEDPDDYDDGFSMKHTATARFQRNHRLISEILSESVVPDVRSVVTTARMQVLKRQVQSLMVHQRKLEAELLQIEERHQEKKRKFLESTDSFNNELKRLCGLKVEVDMEKIAAEIAQAEEQARKRQEEREKEAAEQAERSQSSIVPEEEQAANKGEEKKDDENIPMETEETHLEETTESQQNGEEGTSTPEDKESGQEGVDSMAEEGTSDSNTGSESNSATVEEPPTDPIPEDEKKE</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">L91H</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">0.200196</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">MSKRPSYAPPPTPAPATQMPSTPGFVGYNPYSHLAYNNYRLGGNPGTNSRVTASSGITIPKPPKPPDKPLMPYMRYSRKVWDQVKASNPDLKLWEIGKIIGGMWRDLTDEEKQEYLNEYEAEKIEYNESMKAYHNSPAYLAYINAKSRAEAALEEESRQRQSRMEKGEPYMSIQPAEDPDDYDDGFSMKHTATARFQRNHRLISEILSESVVPDVRSVVTTARMQVLKRQVQSLMVHQRKLEAELLQIEERHQEK</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">MSKRPSYAPPPTPAPATQMPSTPGFVGYNPYSHLAYNNYRLGGNPGTNSRVTASSGITIPKPPKPPDKPLMPYMRYSRKVWDQVKASNPDHKLWEIGKIIGGMWRDLTDEEKQEYLNEYEAEKIEYNESMKAYHNSPAYLAYINAKSRAEAALEEESRQRQSRMEKGEPYMSIQPAEDPDDYDDGFSMKHTATARFQRNHRLISEILSESVVPDVRSVVTTARMQVLKRQVQSLMVHQRKLEAELLQIEERHQEK</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.122331</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">0.200196</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.077865</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">6</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">6</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MLLLHRAVVLRLQQACRLKSIPSRICIQACSTNDSFQPQRPSLTFSGDNSSTQGWRVMGTLLGLGAVLAYQDHRCRAAQESTHIYTKEEVSSHTSPETGIWVTLGSEVFDVTEFVDLHPGGPSKLMLAAGGPLEPFWALYAVHNQSHVRELLAQYKIGELNPEDKVAPTVETSDPYADDPVRHPALKVNSQRPFNAEPPPELLTENYITPNPIFFTRNHLPVPNLDPDTYRLHVVGAPGGQSLSLSLDDLHNFPRYEITVTLQCAGNRRSEMTQVKEVKGLEWRTGAISTARWAGARLCDVLAQAGHQLCETEAHVCFEGLDSDPTGTAYGASIPLARAMDPEAEVLLAYEMNGQPLPRDHGFPVRVVVPGVVGARHVKWLGRVSVQPEESYSHWQRRDYKGFSPSVDWETVDFDSAPSIQELPVQSAITEPRDGETVESGEVTIKGYAWSGGGRAVIRVDVSLDGGLTWQVAKLDGEEQRPRKAWAWRLWQLKAPVPAGQKELNIVCKAVDDGYNVQPDTVAPIWNLRGVLSNAWHRVHVYVSP</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MLLLHRAVVLRLQQACRLKSIPSRICIQACSTNDSFQPQRPSLTFSGDNSSTQGWRVMGTLLGLGAVLAYQDHRCRAAQESTHIYTKEEVSSHTSPETGIWVTLGSEVFDVTEFVDLHPGGPSKLMLAAGGPLEPFWALYAVHNQSHVRECLAQYKIGELNPEDKVAPTVETSDPYADDPVRHPALKVNSQRPFNAEPPPELLTENYITPNPIFFTRNHLPVPNLDPDTYRLHVVGAPGGQSLSLSLDDLHNFPRYEITVTLQCAGNRRSEMTQVKEVKGLEWRTGAISTARWAGARLCDVLAQAGHQLCETEAHVCFEGLDSDPTGTAYGASIPLARAMDPEAEVLLAYEMNGQPLPRDHGFPVRVVVPGVVGARHVKWLGRVSVQPEESYSHWQRRDYKGFSPSVDWETVDFDSAPSIQELPVQSAITEPRDGETVESGEVTIKGYAWSGGGRAVIRVDVSLDGGLTWQVAKLDGEEQRPRKAWAWRLWQLKAPVPAGQKELNIVCKAVDDGYNVQPDTVAPIWNLRGVLSNAWHRVHVYVSP</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">C151L</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.061538</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">RICIQACSTNDSFQPQRPSLTFSGDNSSTQGWRVMGTLLGLGAVLAYQDHRCRAAQESTHIYTKEEVSSHTSPETGIWVTLGSEVFDVTEFVDLHPGGPSKLMLAAGGPLEPFWALYAVHNQSHVRECLAQYKIGELNPEDKVAPTVETSDPYADDPVRHPALKVNSQRPFNAEPPPELLTENYITPNPIFFTRNHLPVPNLDPDTYRLHVVGAPGGQSLSLSLDDLHNFPRYEITVTLQCAGNRRSEMTQVKEV</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">RICIQACSTNDSFQPQRPSLTFSGDNSSTQGWRVMGTLLGLGAVLAYQDHRCRAAQESTHIYTKEEVSSHTSPETGIWVTLGSEVFDVTEFVDLHPGGPSKLMLAAGGPLEPFWALYAVHNQSHVRELLAQYKIGELNPEDKVAPTVETSDPYADDPVRHPALKVNSQRPFNAEPPPELLTENYITPNPIFFTRNHLPVPNLDPDTYRLHVVGAPGGQSLSLSLDDLHNFPRYEITVTLQCAGNRRSEMTQVKEV</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.137421</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.061538</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.075883</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">7</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">7</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAAEKTGIVNVSSSLNVREGASTSSKVIGSLSGNTKVTIVGEEGAFYKIEYKGSHGYVAKEYIKDIKDESAGGSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAAEKTGIVNVSSSLNVREGASTSSKVIGSLSGNTKVTIVGEEGAQYKIEYKGSHGQVAKEYIKDIKDESAGGSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">F44Q:Y55Q</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.664504</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">SAGGSAAEKTGIVNVSSSLNVREGASTSSKVIGSLSGNTKVTIVGEEGAQYKIEYKGSHGQVAKEYIKDIKDESAGGSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">SAGGSAAEKTGIVNVSSSLNVREGASTSSKVIGSLSGNTKVTIVGEEGAFYKIEYKGSHGYVAKEYIKDIKDESAGGSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.572912</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.664504</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.091592</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">8</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">8</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MASGRRAPRTGLLELRCGAGSGAGGERWQRVLLSLAEDALTVSPADGEPGPEPEPAQLNGAAEPGAAPPQLPEALLLQRRRVTVRKADAGGLGISIKGGRENKMPILISKIFKGLAADQTEALFVGDAILSVNGEDLSSATHDEAVQALKKTGKEVVLEVKYWKEVSPYFKNSAGGTSVGWDSPPASPLQRQPSSPGPQPRNLSEAKHVSLKMAYVSRRCTPTDPEPRYLEICAADGQDAVFLRAKDEASARSWAGAIQAQIGTFIPWVKDELQALLTATGTAGSQDIKQIGWLTEQLPSGGTAPTLALLTEKELLFYCSLPQSREALSRPTRTAPLIATSSAHRLVHSGPSKGSVPYDAELSFALRTGTRHGVDTHLFSVESPQELAAWTRQLVDGCHRAAEGIQEVSTACTWNGRPCSLSVHIDKGFTLWAAEPGAARAMLLRQPFEKLQMSSDDGTSLLFLDFGGAEGEIQLDLHSCPKTMVFIIHSFLSAKVTRLGLLA</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MASGRRAPRTGLLELRCGAGSGAGGERWQRVLLSLAEDALTVSPADGEPGPEPEPAQLNGAAEPGAAPPQLPEALLLQRRRVTVRKADAGGLGISIKGGRENKMPILISKIFKGLAADQTEALFVGDAILSVNGEDLSSATHDEAVQALKKTGKEVVLEVKYMKEVSPYFKNSAGGTSVGWDSPPASPLQRQPSSPGPQPRNLSEAKHVSLKMAYVSRRCTPTDPEPRYLEICAADGQDAVFLRAKDEASARSWAGAIQAQIGTFIPWVKDELQALLTATGTAGSQDIKQIGWLTEQLPSGGTAPTLALLTEKELLFYCSLPQSREALSRPTRTAPLIATSSAHRLVHSGPSKGSVPYDAELSFALRTGTRHGVDTHLFSVESPQELAAWTRQLVDGCHRAAEGIQEVSTACTWNGRPCSLSVHIDKGFTLWAAEPGAARAMLLRQPFEKLQMSSDDGTSLLFLDFGGAEGEIQLDLHSCPKTMVFIIHSFLSAKVTRLGLLA</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">M163W</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.04687</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">AEDALTVSPADGEPGPEPEPAQLNGAAEPGAAPPQLPEALLLQRRRVTVRKADAGGLGISIKGGRENKMPILISKIFKGLAADQTEALFVGDAILSVNGEDLSSATHDEAVQALKKTGKEVVLEVKYMKEVSPYFKNSAGGTSVGWDSPPASPLQRQPSSPGPQPRNLSEAKHVSLKMAYVSRRCTPTDPEPRYLEICAADGQDAVFLRAKDEASARSWAGAIQAQIGTFIPWVKDELQALLTATGTAGSQDIKQ</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">AEDALTVSPADGEPGPEPEPAQLNGAAEPGAAPPQLPEALLLQRRRVTVRKADAGGLGISIKGGRENKMPILISKIFKGLAADQTEALFVGDAILSVNGEDLSSATHDEAVQALKKTGKEVVLEVKYWKEVSPYFKNSAGGTSVGWDSPPASPLQRQPSSPGPQPRNLSEAKHVSLKMAYVSRRCTPTDPEPRYLEICAADGQDAVFLRAKDEASARSWAGAIQAQIGTFIPWVKDELQALLTATGTAGSQDIKQ</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.054238</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.04687</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.101108</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">9</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">9</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MQHYGVNGYSLHAMNSLSAMYNLHQQAAQQAQHAPDYRPSVHALTLAERLAGCTFQDIILEARYGSQHRKQRRSRTAFTAQQLEALEKTFQKTHYPDVVMRERLAMCTNLPEARVQVWFKNRRAKFRKKQRSLQKEQLQKQKEAEGSHGEGKAEAPTPDTQLDTEQPPRLPGSDPPAELHLSLSEQSASESAPEDQPDREEDPRAGAEDPKAEKSPGADSKGLGCKRGSPKADSPGSLTITPVAPGGGLLGPSHSYSSSPLSLFRLQEQFRQHMAATNNLVHYSSFEVGGPAPAAAAAAAAVPYLGVNMAPLGSLHCQSYYQSLSAAAAAHQGVWGSPLLPAPPAGLAPASATLNSKTTSIENLRLRAKQHAASLGLDTLPN</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MQHYGVNGYSLHAMNSLSAMYNLHQQAAQQAQHAPDYRPSVHALTLAERLAGCTFQDIILEARYGSQHRKQRRSRTAFTAQQLEALEKTFQKTHYPDVVMRERLAMCTNLPEARVQVWFKVRRAKFRKKQRSLQKEQLQKQKEAEGSHGEGKAEAPTPDTQLDTEQPPRLPGSDPPAELHLSLSEQSASESAPEDQPDREEDPRAGAEDPKAEKSPGADSKGLGCKRGSPKADSPGSLTITPVAPGGGLLGPSHSYSSSPLSLFRLQEQFRQHMAATNNLVHYSSFEVGGPAPAAAAAAAAVPYLGVNMAPLGSLHCQSYYQSLSAAAAAHQGVWGSPLLPAPPAGLAPASATLNSKTTSIENLRLRAKQHAASLGLDTLPN</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">V121N</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.221613</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">MQHYGVNGYSLHAMNSLSAMYNLHQQAAQQAQHAPDYRPSVHALTLAERLAGCTFQDIILEARYGSQHRKQRRSRTAFTAQQLEALEKTFQKTHYPDVVMRERLAMCTNLPEARVQVWFKVRRAKFRKKQRSLQKEQLQKQKEAEGSHGEGKAEAPTPDTQLDTEQPPRLPGSDPPAELHLSLSEQSASESAPEDQPDREEDPRAGAEDPKAEKSPGADSKGLGCKRGSPKADSPGSLTITPVAPGGGLLGPSHS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">MQHYGVNGYSLHAMNSLSAMYNLHQQAAQQAQHAPDYRPSVHALTLAERLAGCTFQDIILEARYGSQHRKQRRSRTAFTAQQLEALEKTFQKTHYPDVVMRERLAMCTNLPEARVQVWFKNRRAKFRKKQRSLQKEQLQKQKEAEGSHGEGKAEAPTPDTQLDTEQPPRLPGSDPPAELHLSLSEQSASESAPEDQPDREEDPRAGAEDPKAEKSPGADSKGLGCKRGSPKADSPGSLTITPVAPGGGLLGPSHS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.153544</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.221613</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.068069</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">10</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">10</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MAAAVRMNIQMLLEAADYLERREREAEHGYASMLPYNNKDRDALKRRNKSKKNNSSSRSTHNEMEKNRRAHLRLCLEKLKGLVPLGPESWRHTTLSLLTKAKLHIKKLEDCDRKAVHQIDQLQREQRHLKRQLEKLGIERIRMDSIGSTVSSERSDSDREEIDVDVESTDYLTGDLDWSSSSVSDSDERGSMQSLGSDEGYSSTSIKRIKLQDSHKACLGL</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MAAAVRMNIQMLLEAADYLERREREAEHGYASMLPYNNKDRDALKRRNKSKKNNSSSRSTHNEMEKNRRAHLRLCLEKLKGLVPLGPESSRHTTLSLLTKAKLHIKKLEDCDRKAVHQIDQLQREQRHLKRQLEKLGIERIRMDSIGSTVSSERSDSDREEIDVDVESTDYLTGDLDWSSSSVSDSDERGSMQSLGSDEGYSSTSIKRIKLQDSHKACLGL</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">S90W</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.089817</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">MAAAVRMNIQMLLEAADYLERREREAEHGYASMLPYNNKDRDALKRRNKSKKNNSSSRSTHNEMEKNRRAHLRLCLEKLKGLVPLGPESSRHTTLSLLTKAKLHIKKLEDCDRKAVHQIDQLQREQRHLKRQLEKLGIERIRMDSIGSTVSSERSDSDREEIDVDVESTDYLTGDLDWSSSSVSDSDERGSMQSLGSDEGYSSTSIKRIKLQDSHKACLGL</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">MAAAVRMNIQMLLEAADYLERREREAEHGYASMLPYNNKDRDALKRRNKSKKNNSSSRSTHNEMEKNRRAHLRLCLEKLKGLVPLGPESWRHTTLSLLTKAKLHIKKLEDCDRKAVHQIDQLQREQRHLKRQLEKLGIERIRMDSIGSTVSSERSDSDREEIDVDVESTDYLTGDLDWSSSSVSDSDERGSMQSLGSDEGYSSTSIKRIKLQDSHKACLGL</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.111485</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.089817</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.201302</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">11</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">11</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MKAAVDLKPTLTIIKTEKVDLELFPSPDMECADVPLLTPSSKEMMSQALKATFSGFTKEQQRLGIPKDPRQWTETHVRDWVMWAVNEFYLKGVDFQKFCMNGAALCALGKDCFLELAPDFVGDILWEHLEILQKEDVKPYQVNGVNPAYPESRYTSDYFISYGIEHAQCVPPSEFSEPSFITESYQTLHPISSEELLSLKYENDYPSVILRDPLQTDTLQNDYFAIKQEVVTPDNMCMGRTSRGKLGGQDSFESIESYDSCDRLTQSWSSQSSFNSLQRVPSYDSFDSEDYPAALPNHKPKGTFKDYVRDRADLNKDKPVIPAAALAGYTGSGPIQLWQFLLELLTDKSCQSFISWTGDGWEFKLSDPDEVARRWGKRKNKPKMNYEKLSRGLRYYYDKNIIHKTAGKRYVYRFVCDLQSLLGYTPEELHAMLDVKPDADE</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MKAAVDLKPTLTIIKTEKVDLELFPSPDMECADVPLLTPSSKEMMSQALKATFSGFTKEQQRLGIPKDPRQWTETHVRDWVMWAVNEFSLKGVDFQKFCMNGAALCALGKDCFLELAPDFVGDILWEHLEILQKEDVKPYQVNGVNPAYPESRYTSDYFISYGIEHAQCVPPSEFSEPSFITESYQTLHPISSEELLSLKYENDYPSVILRDPLQTDTLQNDYFAIKQEVVTPDNMCMGRTSRGKLGGQDSFESIESYDSCDRLTQSWSSQSSFNSLQRVPSYDSFDSEDYPAALPNHKPKGTFKDYVRDRADLNKDKPVIPAAALAGYTGSGPIQLWQFLLELLTDKSCQSFISWTGDGWEFKLSDPDEVARRWGKRKNKPKMNYEKLSRGLRYYYDKNIIHKTAGKRYVYRFVCDLQSLLGYTPEELHAMLDVKPDADE</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">S89Y</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">0.124885</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">MKAAVDLKPTLTIIKTEKVDLELFPSPDMECADVPLLTPSSKEMMSQALKATFSGFTKEQQRLGIPKDPRQWTETHVRDWVMWAVNEFSLKGVDFQKFCMNGAALCALGKDCFLELAPDFVGDILWEHLEILQKEDVKPYQVNGVNPAYPESRYTSDYFISYGIEHAQCVPPSEFSEPSFITESYQTLHPISSEELLSLKYENDYPSVILRDPLQTDTLQNDYFAIKQEVVTPDNMCMGRTSRGKLGGQDSFESI</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">MKAAVDLKPTLTIIKTEKVDLELFPSPDMECADVPLLTPSSKEMMSQALKATFSGFTKEQQRLGIPKDPRQWTETHVRDWVMWAVNEFYLKGVDFQKFCMNGAALCALGKDCFLELAPDFVGDILWEHLEILQKEDVKPYQVNGVNPAYPESRYTSDYFISYGIEHAQCVPPSEFSEPSFITESYQTLHPISSEELLSLKYENDYPSVILRDPLQTDTLQNDYFAIKQEVVTPDNMCMGRTSRGKLGGQDSFESI</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.062564</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">0.124885</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.062321</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">12</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">12</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MGSKGVYQYHWQSHNVKHSGVDDMVLLSKITENSIVENLKKRYMDDYIFTYIGSVLISVNPFKQMPYFGEKEIEMYQGAAQYENPPHIYALADNMYRNMIIDRENQCVIISGESGAGKTVAAKYIMSYISRVSGGGTKVQHVKDIILQSNPLLEAFGNAKTVRNNNSSRFGKYFEIQFSPGGEPDGGKISNFLLEKSRVVMRNPGERSFHIFYQLIEGASAEQKHSLGITSMDYYYYLSLSGSYKVDDIDDRREFQETLHAMNVIGIFAEEQTLVLQIVAGILHLGNISFKEVGNYAAVESEEFLAFPAYLLGINQDRLKEKLTSRQMDSKWGGKSESIHVTLNVEQACYTRDALAKALHARVFDFLVDSINKAMEKDHEEYNIGVLDIYGFEIFQKNGFEQFCINFVNEKLQQIFIELTLKAEQEEYVQEGIRWTPIEYFNNKIVCDLIENKVNPPGIMSILDDVCATMHAVGEGADQTLLQKLQMQIGSHEHFNSWNQGFIIHHYAGKVSYDMDGFCERNRDVLFMDLIELMQSSELPFIKSLFPENLQADKKGRPTTAGSKIKKQANDLVSTLMKCTPHYIRCIKPNETKKPRDWEESRVKHQVEYLGLKENIRVRRAGYAYRRIFQKFLQRYAILTKATWPSWQGEEKQGVLHLLQSVNMDSDQFQLGRSKVFIKAPESLFLLEEMRERKYDGYARVIQKSWRKFVARKKYVQMREEASDLLLNKKERRRNSINRNFIGDYIGMEEHPELQQFVGKREKIDFADTVTKYDRRFKGVKRDLLLTPKCLYLIGREKVKQGPDKGLVKEVLKRKIEIERILSVSLSTMQDDIFILHEQEYDSLLESVFKTEFLSLLAKRYEEKTQKQLPLKFSNTLELKLKKENWGPWSAGGSRQVQFHQGFGDLAVLKPSNKVLQVSIGPGLPKNSRPTRRNTTQNTGYSSGTQNANYPVRAAPPPPGYHQNGVIRNQYVPYPHAPGSQRSNQKSLYTSMARPPLPRQQSTSSDRVSQTPESLDFLKVPDQGAAGVRRQTTSRPPPAGGRPKPQPKPKPQVPQCKALYAYDAQDTSELSFNANDIIDIIKEDPSGWWTGRLRGKQGLFPNNYVTKI</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MGSKGVYQYHWQSHNVKHSGVDDMVLLSKITENSIVENLKKRYMDDYIFTYIGSVLISVNPFKQMPYFGEKEIEMYQGAAQYENPPHIYALADNMYRNMIIDRENQCVIISGESGAGKTVAAKYIMSYISRVSGGGTKVQHVKDIILQSNPLLEAFGNAKTVRNNNSSRFGKYFEIQFSPGGEPDGGKISNFLLEKSRVVMRNPGERSFHIFYQLIEGASAEQKHSLGITSMDYYYYLSLSGSYKVDDIDDRREFQETLHAMNVIGIFAEEQTLVLQIVAGILHLGNISFKEVGNYAAVESEEFLAFPAYLLGINQDRLKEKLTSRQMDSKWGGKSESIHVTLNVEQACYTRDALAKALHARVFDFLVDSINKAMEKDHEEYNIGVLDIYGFEIFQKNGFEQFCINFVNEKLQQIFIELTLKAEQEEYVQEGIRWTPIEYFNNKIVCDLIENKVNPPGIMSILDDVCATMHAVGEGADQTLLQKLQMQIGSHEHFNSWNQGFIIHHYAGKVSYDMDGFCERNRDVLFMDLIELMQSSELPFIKSLFPENLQADKKGRPTTAGSKIKKQANDLVSTLMKCTPHYIRCIKPNETKKPRDWEESRVKHQVEYLGLKENIRVRRAGYAYRRIFQKFLQRYAILTKATWPSWQGEEKQGVLHLLQSVNMDSDQFQLGRSKVFIKAPESLFLLEEMRERKYDGYARVIQKSWRKFVARKKYVQMREEASDLLLNKKERRRNSINRNFIGDYIGMEEHPELQQFVGKREKIDFADTVTKYDRRFKGVKRDLLLTPKCLYLIGREKVKQGPDKGLVKEVLKRKIEIERILSVSLSTMQDDIFILHEQEYDSLLESVFKTEFLSLLAKRYEEKTQKQLPLKFSNTLELKLKKENWGPWSAGGSRQVQFHQGFGDLAVLKPSNKVLQVSIGPGLPKNSRPTRRNTTQNTGYSSGTQNANYPVRAAPPPPGYHQNGVIRNQYVPYPHAPGSQRSNQKSLYTSMARPPLPRQQSTSSDRVSQTPESLDFLKVPDQGAAGVRRQTTSRPPPAGGRPKPQPKPKPQVPQCKALYAYDAQDTDELSFNANDIIDIIKEDPSGWWTGRLRGKQGLFPNNYVTKI</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">D1068S</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.003785</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">LSLLAKRYEEKTQKQLPLKFSNTLELKLKKENWGPWSAGGSRQVQFHQGFGDLAVLKPSNKVLQVSIGPGLPKNSRPTRRNTTQNTGYSSGTQNANYPVRAAPPPPGYHQNGVIRNQYVPYPHAPGSQRSNQKSLYTSMARPPLPRQQSTSSDRVSQTPESLDFLKVPDQGAAGVRRQTTSRPPPAGGRPKPQPKPKPQVPQCKALYAYDAQDTDELSFNANDIIDIIKEDPSGWWTGRLRGKQGLFPNNYVTKI</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">LSLLAKRYEEKTQKQLPLKFSNTLELKLKKENWGPWSAGGSRQVQFHQGFGDLAVLKPSNKVLQVSIGPGLPKNSRPTRRNTTQNTGYSSGTQNANYPVRAAPPPPGYHQNGVIRNQYVPYPHAPGSQRSNQKSLYTSMARPPLPRQQSTSSDRVSQTPESLDFLKVPDQGAAGVRRQTTSRPPPAGGRPKPQPKPKPQVPQCKALYAYDAQDTSELSFNANDIIDIIKEDPSGWWTGRLRGKQGLFPNNYVTKI</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.010197</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.003785</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.006412</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">13</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">13</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">SATKAQVIEAFKVFDRDGNGYVTVDYLRKVLNELGDMMPADEIEEMIYEADPQNSGYVQYETFVGMLFLSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">SATKAQVIEAFKVFDRDGNGYVTVDYLRKVQNELGDMMPADEIEEMIYEADPQNSGYVQYETFVGMLFLSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">L29Q</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.249197</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">SATKAQVIEAFKVFDRDGNGYVTVDYLRKVQNELGDMMPADEIEEMIYEADPQNSGYVQYETFVGMLFLSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">SATKAQVIEAFKVFDRDGNGYVTVDYLRKVLNELGDMMPADEIEEMIYEADPQNSGYVQYETFVGMLFLSAG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.201009</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.249197</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.048188</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">14</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">14</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">SAGGSPAHPYDNLKTTSTEPVSDIDVTRREAYLSSEEFKEKFGMTKEAFYKLPKWKQNKFKMAVQLFSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">SAGGSPAHPYDRLKTTSTDPVSDIDVTRREAYLSSEEFKEKFGMTKEAFYKLPKWKQNKFKMAVQLFSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">R7N:D14E</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">0.043277</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">SAGGSPAHPYDRLKTTSTDPVSDIDVTRREAYLSSEEFKEKFGMTKEAFYKLPKWKQNKFKMAVQLFSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">SAGGSPAHPYDNLKTTSTEPVSDIDVTRREAYLSSEEFKEKFGMTKEAFYKLPKWKQNKFKMAVQLFSAGGS</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.008795</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">0.043277</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.034483</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">15</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">15</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">GGLEHMADEEKLPPGWEKRMERSSGRVYYFNHITNASQWERPSG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">GGLEHMADEEKLPPGWEKRMERSSGRVYYFNHITNASQWERQSG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">P41Q</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.336373</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">GGLEHMADEEKLPPGWEKRMERSSGRVYYFNHITNASQWERQSG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">GGLEHMADEEKLPPGWEKRMERSSGRVYYFNHITNASQWERPSG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.127705</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.336373</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.208668</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">16</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">16</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGRLMNAFMVWARIHRPALAKANPAANNAEISVQLGLEWNKLSEEQKKPYYDEAQKIKESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGRPMNAFMVWARIHRPALAKANPAANNAEISVQLGLEWNKLSEEQKKPYYDEAQKIKESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">P2L</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.044278</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">SAGGSAGRPMNAFMVWARIHRPALAKANPAANNAEISVQLGLEWNKLSEEQKKPYYDEAQKIKESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">SAGGSAGRLMNAFMVWARIHRPALAKANPAANNAEISVQLGLEWNKLSEEQKKPYYDEAQKIKESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">0.138658</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.044278</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.182937</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">17</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">17</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MHKHQHCCKCPECYEVTRLAALRRLEPPGYGDWQVPDPYGPGGGNGASAGYGGYSSQTLPSQAGATPTPRTKAKLIPTGRDVGPVPPKPVPGKSTPKLNGSGPSWWPECTCTNRDWYEQVNGSDGMFKYEEIVLERGNSGLGFSIAGGIDNPHVPDDPGIFITKIIPGGAAAMDGRLGVNDCVLRVNEVDVSEVVHSRAVEALKEAGPVVRLVVRRRQPPPETIMEVNLLKGPKGLGFSIAGGIGNQHIPGDNSIYITKIIEGGAAQKDGRLQIGDRLLAVNNTNLQDVRHEEAVASLKNTSDMVYLKVAKPGSLHLNDMYAPPDYASTFTALADNHISHNSSLGYLGAVESKVSYPAPPQVPPTRYSPIPRHMLAEEDFTREPRKIILHKGSTGLGFNIVGGEDGEGIFVSFIGAGGPADLSGELRRGDRILSVNGVNLRNATHEQAAAALKRAGQSVTIVAQYRPEEYSRFESKIHDLREQMMNSSMSSGSGSLRTSEKRSLYVRALFDYDRTRDSCLPSQGLSFSYGDILHVINASDDEWWQARLVTPHGESEQIGVIPSKKRVEKKERARLKTVKFHARTGMIESNRDFPGLSDDYYGAKNLKGQEDAILSYEPVTRQEIHYARPVIILGPMKDRVNDDLISEFPHKFGSCVPHTTRPRRDNEVDGQDYHFVVSREQMEKDIQDNKFIEAGQFNDNLYGTSIQSVRAVAERGKHCILDVSGNAIKRLQQAQLYPIAIFIKPKSIEALMEMNRRQTYEQANKIYDKAMKLEQEFGEYFTAIVQGDSLEEIYNKIKQIIEDQSGHYIWVPSPEKL</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MHKHQHCCKCPECYEVTRLAALRRLEPPGYGDWQVPDPYGPGGGNGASAGYGGYSSQTLPSQAGATPTPRTKAKLIPTGRDVGPVPPKPVPGKSTPKLNGSGPSWWPECTCTNRDWYEQVNGSDGMFKYEEIVLERGNSGLGFSIAGGIDNPHVPDDPGIFITKIIPGGAAAMDGRLGVNDCVLRVNEVDVSEVVHSRAVEALKEAGPVVRLVVRRRQPPPETIMEVNLLKGPKGLGFSIAGGIGNQHIPGDNSIYITKIIEGGAAQKDGRLQIGDRLLAVNNTNLQDVRHEEAVASLKNTSDMVYLKVAKPGSLHLNDMYAPPDYASTFTALADNHISHNSSLGYLGAVESKVSYPAPPQVPPTRYSPIPRHMLAEEDFTREPRKIILHKGSTGLGFNIVGGEDGEGIFVSFILAGGPADLSGELRRGDRILSVNGVNLRNATHEQAAAALKRAGQSVTIVAQYRPEEYSRFESKIHDLREQMMNSSMSSGSGSLRTSEKRSLYVRALFDYDRTRDSCLPSQGLSFSYGDILHVINASDDEWWQARLVTPHGESEQIGVIPSKKRVEKKERARLKTVKFHARTGMIESNRDFPGLSDDYYGAKNLKGQEDAILSYEPVTRQEIHYARPVIILGPMKDRVNDDLISEFPHKFGSCVPHTTRPRRDNEVDGQDYHFVVSREQMEKDIQDNKFIEAGQFNDNLYGTSIQSVRAVAERGKHCILDVSGNAIKRLQQAQLYPIAIFIKPKSIEALMEMNRRQTYEQANKIYDKAMKLEQEFGEYFTAIVQGDSLEEIYNKIKQIIEDQSGHYIWVPSPEKL</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">L415G</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">0.091064</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">DVRHEEAVASLKNTSDMVYLKVAKPGSLHLNDMYAPPDYASTFTALADNHISHNSSLGYLGAVESKVSYPAPPQVPPTRYSPIPRHMLAEEDFTREPRKIILHKGSTGLGFNIVGGEDGEGIFVSFILAGGPADLSGELRRGDRILSVNGVNLRNATHEQAAAALKRAGQSVTIVAQYRPEEYSRFESKIHDLREQMMNSSMSSGSGSLRTSEKRSLYVRALFDYDRTRDSCLPSQGLSFSYGDILHVINASDDE</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">DVRHEEAVASLKNTSDMVYLKVAKPGSLHLNDMYAPPDYASTFTALADNHISHNSSLGYLGAVESKVSYPAPPQVPPTRYSPIPRHMLAEEDFTREPRKIILHKGSTGLGFNIVGGEDGEGIFVSFIGAGGPADLSGELRRGDRILSVNGVNLRNATHEQAAAALKRAGQSVTIVAQYRPEEYSRFESKIHDLREQMMNSSMSSGSGSLRTSEKRSLYVRALFDYDRTRDSCLPSQGLSFSYGDILHVINASDDE</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.030392</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">0.091064</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.121456</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-odd" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">18</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">18</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">MGNCAKRPWRRGPKDPLQWLGSPPRGSCPSPSSSPKEQGDPAPGVQGYSVLNSLVGPACIFLRPSIAATQLDRELRPEEIEELQVAFQEFDRDRDGYIGCRELGACMRTLGYMPTEMELIEISQQISGGKVDFEDFVELMGPKLLAETADMIGVRELRDAFREFDTNGDGRISVGELRAALKALLGERLSQREVDEILLDVDLNGDGLVDFEEFVRMMSR</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">MGNCAKRPWRRGPKDPLQWLGSPPRGSCPSPSSSPKEQGDPAPGVQGYSVLNSLVGPACIFLRPSIAATQLDRELRPEEIEELQVAFQEFDRDRDGYIGCRELGACMRTLGYMPTEMELIEISQQISGGKVDFEDFVELMGPKLLAETADMIGVRELRDAFREFDTNGDGRISVGELRAALKALLGERLSQREVDEILQDVDLNGDGLVDFEEFVRMMSR</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">Q199L</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.006505</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">true</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">lehner</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">MGNCAKRPWRRGPKDPLQWLGSPPRGSCPSPSSSPKEQGDPAPGVQGYSVLNSLVGPACIFLRPSIAATQLDRELRPEEIEELQVAFQEFDRDRDGYIGCRELGACMRTLGYMPTEMELIEISQQISGGKVDFEDFVELMGPKLLAETADMIGVRELRDAFREFDTNGDGRISVGELRAALKALLGERLSQREVDEILQDVDLNGDGLVDFEEFVRMMSR</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">MGNCAKRPWRRGPKDPLQWLGSPPRGSCPSPSSSPKEQGDPAPGVQGYSVLNSLVGPACIFLRPSIAATQLDRELRPEEIEELQVAFQEFDRDRDGYIGCRELGACMRTLGYMPTEMELIEISQQISGGKVDFEDFVELMGPKLLAETADMIGVRELRDAFREFDTNGDGRISVGELRAALKALLGERLSQREVDEILLDVDLNGDGLVDFEEFVRMMSR</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.006958</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.006505</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">-0.000454</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div><div class="tabulator-row tabulator-selectable tabulator-row-even" role="row"><div class="tabulator-cell tabulator-frozen tabulator-frozen-left" role="gridcell" tabulator-field="_index" style="display: none; position: sticky; left: 0px; height: 28px;">19</div><div class="tabulator-cell" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="index"><div style="text-align: left;">19</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="original_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGMTTFKLIINGKTLKGEITIEAVDAAEAEKIFKQYANDNGIDGEWTYDDATKTDTVTESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mutated_seq_full" tabindex="0"><div style="text-align: left;">SAGGSAGMTTFKLIINGKTLKGEITIEAVDAKEAEKIFKQYANDNGIDGEWTYDDATKTDTVTESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="mut_type" tabindex="0"><div style="text-align: left;">A25K</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="target" tabindex="0"><div style="text-align: right;">-0.018531</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="reverse" tabindex="0"><div style="text-align: center;">false</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="data_source" tabindex="0"><div style="text-align: left;">megascale</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_mut" tabindex="0"><div style="text-align: left;">SAGGSAGMTTFKLIINGKTLKGEITIEAVDAKEAEKIFKQYANDNGIDGEWTYDDATKTDTVTESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="fragment_255_org" tabindex="0"><div style="text-align: left;">SAGGSAGMTTFKLIINGKTLKGEITIEAVDAAEAEKIFKQYANDNGIDGEWTYDDATKTDTVTESAGGSAGG</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="predicted_fitness" tabindex="0"><div style="text-align: right;">-0.003642</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 165px; height: 28px;" tabulator-field="actual_fitness" tabindex="0"><div style="text-align: right;">-0.018531</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell tabulator-editable" role="gridcell" style="width: 166px; height: 28px;" tabulator-field="error" tabindex="0"><div style="text-align: right;">0.014889</div></div><span class="tabulator-col-resize-handle"></span><div class="tabulator-cell empty" role="gridcell" style="width: 1px; height: 28px;">&nbsp;</div></div></div></div><div class="tabulator-footer"><div class="tabulator-footer-contents"><span class="tabulator-paginator"><button class="tabulator-page" type="button" role="button" aria-label="First Page" title="First Page" data-page="first" disabled="">First</button><button class="tabulator-page" type="button" role="button" aria-label="Prev Page" title="Prev Page" data-page="prev" disabled="">Prev</button><span class="tabulator-pages"><button class="tabulator-page active" type="button" role="button" aria-label="Show Page 1" title="Show Page 1" data-page="1">1</button><button class="tabulator-page" type="button" role="button" aria-label="Show Page 2" title="Show Page 2" data-page="2">2</button><button class="tabulator-page" type="button" role="button" aria-label="Show Page 3" title="Show Page 3" data-page="3">3</button><button class="tabulator-page" type="button" role="button" aria-label="Show Page 4" title="Show Page 4" data-page="4">4</button><button class="tabulator-page" type="button" role="button" aria-label="Show Page 5" title="Show Page 5" data-page="5">5</button></span><button class="tabulator-page" type="button" role="button" aria-label="Next Page" title="Next Page" data-page="next">Next</button><button class="tabulator-page" type="button" role="button" aria-label="Last Page" title="Last Page" data-page="last">Last</button></span></div></div></div>
"""

# Zkontrolujte, zda byl HTML kód vložen, jinak vypište zprávu
if "PASTE YOUR HTML HERE" in html_code or not html_code.strip():
    print("Prosím, vložte svůj HTML kód do proměnné 'html_code'.")
else:
    # Zavolání funkce pro vytvoření DataFrame
    df = html_table_to_dataframe(html_code)

    # Zobrazení prvních 5 řádků a informací o DataFrame
    print("--- Prvních 5 řádků DataFrame: ---")
    print(df.head())

    print("\n--- Informace o DataFrame (datové typy sloupců): ---")
    df.info()

--- Prvních 5 řádků DataFrame: ---
   index                                  original_seq_full  \
0      0  SAGGSAGGSAGGSAGGSDLRKKIVDLHKSGSSLGAISKRLKVPRSS...   
1      1  MPSGSSAALALAAAPAPLPQPPPPPPPPPPPLPPPSGGPELEGDGL...   
2      2  SAGGSAGGKFNKELSVAGREIVTLPNLNDPQKKAFIFSLWDDPSQS...   
3      3  MAALSGGGGGGAEPGQALFNGDMEPEAGAGAGAAASSAADPAIPEE...   
4      4  MEAERRRQAEKPKKGRVGSNLLPERHPATGTPTTTVDSSAPPCRRL...   

                                    mutated_seq_full   mut_type    target  \
0  SAGGSAGGSAGGSAGGSDLRKKIVDLHKSGSSLGAISKRLKVPRSS...       Q31D  0.117479   
1  MPSGSSAALALAAAPAPLPQPPPPPPPPPPPLPPPSGGPELEGDGL...      D366P  0.414292   
2  SAGGSAGGKFNKELSVAGREIVTLPNLNDPQKKAFIFSLWDWPFQS...  D34W:S36F -0.171998   
3  MAALSGGGGGGAEPGQALFNGDMEPEAGAGAGAAASSAADPAIPEE...      E210W -0.055847   
4  MEAERRRQAEKPKKGRVGSNLLPERHPATGTPTTTVDSSAPPCRRL...      Q114V  0.046613   

   reverse data_source                                   fragment_255_mut  \
0     True   megascale  SAGGSAGGSAGGSAGGSDLRKK

/tmp/ipykernel_71891/2416398181.py:53: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [45]:
# save to file

import os

with open("results_eval.html", "w") as f:
    f.write(log_interactive_js_table_to_wandb(df_duplicated, "results_eval"))


Finální funkční soubor uložen jako: results_eval_final_working.html


In [44]:
list_of_dfs = [df] * 10000

# Spojíme všechny DataFrame v seznamu do jednoho velkého
df_duplicated = pd.concat(list_of_dfs, ignore_index=True)